# Window-by-Window Inference Demo

Notebook to reproduce the near–real-time inference workflow:
1. Select the trained experiment and an enriched_*.parquet session.
2. Run run_inference.py, which applies the pipeline on the fly and saves per-window predictions.
3. Analyze metrics and visualize fatigue_pred vs. fatigue_score to document the model's behavior.

### 1. Configuration

Adjust the paths according to the experiment/session you want to evaluate.

In [42]:
from pathlib import Path

EXPERIMENT_DIR = Path("../data/results/modeling/experiments/runner_id_20251119_193024")
MODEL_NAME = "gradient_boosting"
ENRICHED_PATH = Path("../data/enriched/enriched_D_231001_runTEST__3_184.0_LPM_2025_10_24_13_06_22_T3_1.parquet")
OUTPUT_PATH = Path("../data/results/modeling/inference/demo_predictions_notebook.parquet")
WINDOW_SECONDS = 3.0
OVERLAP_RATIO = 0.75
PLAYBACK_SPEED = 0.0

EXPERIMENT_DIR, ENRICHED_PATH, OUTPUT_PATH

(PosixPath('../data/results/modeling/experiments/runner_id_20251119_193024'),
 PosixPath('../data/enriched/enriched_D_231001_runTEST__3_184.0_LPM_2025_10_24_13_06_22_T3_1.parquet'),
 PosixPath('../data/results/modeling/inference/demo_predictions_notebook.parquet'))

### 2. Run run_inference.py

The script reuses the saved pipeline and generates window-level predictions.

In [43]:
import subprocess
import shlex

cmd = [
    "python",
    "../src/models/run_inference.py",
    "--enriched", str(ENRICHED_PATH),
    "--experiment", str(EXPERIMENT_DIR),
    "--model", MODEL_NAME,
    "--window", str(WINDOW_SECONDS),
    "--overlap", str(OVERLAP_RATIO),
    "--output", str(OUTPUT_PATH),
    "--playback-speed", str(PLAYBACK_SPEED),
]
print("Comando:", " ".join(shlex.quote(part) for part in cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:\n", result.stdout)
print("STDERR:\n", result.stderr)
result.check_returncode()

Comando: python ../src/models/run_inference.py --enriched ../data/enriched/enriched_D_231001_runTEST__3_184.0_LPM_2025_10_24_13_06_22_T3_1.parquet --experiment ../data/results/modeling/experiments/runner_id_20251119_193024 --model gradient_boosting --window 3.0 --overlap 0.75 --output ../data/results/modeling/inference/demo_predictions_notebook.parquet --playback-speed 0.0
STDOUT:
 
STDERR:
 2025-11-20 10:22:19,512 - INFO - Evaluation (fatigue_score per ventana) -> MAE=0.0646 RMSE=0.0741 R2=0.8277
2025-11-20 10:22:19,515 - INFO - [t=  0.00s] pred=0.610 | score=0.516
2025-11-20 10:22:19,515 - INFO - [t=  0.76s] pred=0.948 | score=0.921
2025-11-20 10:22:19,515 - INFO - [t=  1.51s] pred=0.935 | score=0.908
2025-11-20 10:22:19,515 - INFO - [t=  2.26s] pred=0.935 | score=0.908
2025-11-20 10:22:19,515 - INFO - [t=  3.01s] pred=0.939 | score=0.901
2025-11-20 10:22:19,515 - INFO - [t=  3.76s] pred=0.602 | score=0.498
2025-11-20 10:22:19,515 - INFO - [t=  4.50s] pred=0.599 | score=0.510
2025-11

### 3. Load predictions and review metrics

Use this step to verify the model’s performance and ensure the inference behaves consistently with training results.

In [44]:
import pandas as pd
import numpy as np

pred_df = pd.read_parquet(OUTPUT_PATH)
pred_df.head()


,file,source_file,start_s,duration,n_samples,acc_x_centered_mean,acc_x_centered_std,acc_x_centered_mad,acc_x_centered_skew,acc_x_centered_kurt,...,grav_z_skew,grav_z_kurt,jerk_mean,jerk_std,jerk_mad,jerk_skew,fc_mean,spo2_mean,fatigue_score,fatigue_pred
0,clean_D_231001_runTEST__3_184.0_LPM_2025_10_24...,enriched_D_231001_runTEST__3_184.0_LPM_2025_10...,0.000000,2.996452,301,0.029476,0.811144,0.556568,-0.120877,-1.303989,...,-0.447481,-0.820043,32.156627,24.086299,15.790979,1.216937,177.0,99.0,0.516,0.610153
1,clean_D_231001_runTEST__3_184.0_LPM_2025_10_24...,enriched_D_231001_runTEST__3_184.0_LPM_2025_10...,0.759215,2.986286,300,0.050149,0.810326,0.576823,-0.163988,-1.260210,...,-0.113550,-0.921474,93.903161,398.618975,15.656637,6.686746,177.0,99.0,0.921,0.947705
2,clean_D_231001_runTEST__3_184.0_LPM_2025_10_24...,enriched_D_231001_runTEST__3_184.0_LPM_2025_10...,1.508280,2.986378,300,0.037904,0.766258,0.578933,-0.219545,-1.482254,...,0.019903,-0.990459,91.730517,398.834136,14.166484,6.692005,177.0,99.0,0.908,0.934759
3,clean_D_231001_runTEST__3_184.0_LPM_2025_10_24...,enriched_D_231001_runTEST__3_184.0_LPM_2025_10...,2.257369,2.989729,300,0.039549,0.778495,0.589308,-0.267555,-1.513535,...,0.085947,-0.431404,91.577013,398.917432,14.120138,6.688867,177.0,99.0,0.908,0.935011
4,clean_D_231001_runTEST__3_184.0_LPM_2025_10_24...,enriched_D_231001_runTEST__3_184.0_LPM_2025_10...,3.006426,2.986345,300,0.046608,0.803415,0.576618,-0.297721,-1.488704,...,0.210968,-0.035865,91.811495,398.899950,13.179422,6.688007,177.0,99.0,0.901,0.938615


In [45]:
metrics = {}
if "fatigue_score" in pred_df.columns and not pred_df["fatigue_score"].isna().all():
    y_true = pred_df["fatigue_score"].to_numpy()
    y_pred = pred_df["fatigue_pred"].to_numpy()
    from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
    metrics = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred, squared=False),
        "R2": r2_score(y_true, y_pred),
    }
metrics

{'MAE': 0.064556136937306,
 'RMSE': 0.07410687035232409,
 'R2': 0.8276577463445076}

### 4. Visualizations

Temporal comparison and scatter analysis of fatigue_pred vs. fatigue_score.

In [46]:
import plotly.express as px

if "fatigue_score" in pred_df.columns and not pred_df["fatigue_score"].isna().all():
    long_df = pred_df.melt(
        id_vars=["start_s"],
        value_vars=["fatigue_pred", "fatigue_score"],
        var_name="serie",
        value_name="valor",
    )
else:
    long_df = pred_df[["start_s", "fatigue_pred"]].assign(serie="fatigue_pred", valor=pred_df["fatigue_pred"])

fig = px.line(long_df, x="start_s", y="valor", color="serie", title="Predicción vs. score por ventana")
fig.update_layout(xaxis_title="Tiempo (s)", yaxis_title="Fatigue score")
fig.show()


In [47]:
if "fatigue_score" in pred_df.columns and not pred_df["fatigue_score"].isna().all():
    fig = px.scatter(
        pred_df,
        x="fatigue_score",
        y="fatigue_pred",
        title="Dispersión score real vs. predicho",
        labels={"fatigue_score": "Score real", "fatigue_pred": "Score predicho"},
    )
    fig.add_shape(
        type="line",
        x0=pred_df["fatigue_score"].min(),
        x1=pred_df["fatigue_score"].max(),
        y0=pred_df["fatigue_score"].min(),
        y1=pred_df["fatigue_score"].max(),
        line=dict(color="gray", dash="dash"),
    )
    fig.show()
else:
    print("No hay score real en el archivo; solo se muestra la serie predicha.")
